# D1 Utilities Workflow Notebook

Run this notebook top-to-bottom. It has two tracks:
1. Simple track
2. Complex track

All API calls are connector-based with offline fallback so every section is runnable.


In [7]:
from __future__ import annotations

import json
import math
import os
import random
from heapq import heappop, heappush
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

import pandas as pd

OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_LIVE_API = os.getenv('GEOPROMPT_ALLOW_LIVE_API', '0') == '1'


def fetch_json(url: str, fallback: dict) -> dict:
    if not ALLOW_LIVE_API:
        return fallback
    try:
        req = Request(url, headers={'User-Agent': 'geoprompt-section-d-notebook/2.0'})
        with urlopen(req, timeout=10) as response:  # nosec B310
            return json.loads(response.read().decode('utf-8'))
    except (URLError, TimeoutError, ValueError):
        return fallback


def write_svg_bar(path: Path, title: str, rows: list[tuple[str, float]], color: str = '#2563eb') -> Path:
    width = 980
    height = max(280, 80 + len(rows) * 28)
    pad_left = 260
    pad_top = 48
    bar_h = 18
    gap = 10
    max_v = max([v for _, v in rows], default=1.0)

    lines = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<style>text{font-family:Segoe UI,Arial,sans-serif;font-size:12px;fill:#1f2937}.title{font-size:20px;font-weight:700}</style>',
        f'<rect width="{width}" height="{height}" fill="#f8fafc"/>',
        f'<text x="20" y="30" class="title">{title}</text>',
    ]
    for i, (label, value) in enumerate(rows):
        y = pad_top + i * (bar_h + gap)
        bar_w = int((value / max(max_v, 1e-9)) * (width - pad_left - 60))
        lines.append(f'<text x="18" y="{y + 13}">{label}</text>')
        lines.append(f'<rect x="{pad_left}" y="{y}" width="{bar_w}" height="{bar_h}" rx="3" fill="{color}" opacity="0.9"/>')
        lines.append(f'<text x="{pad_left + bar_w + 8}" y="{y + 13}">{value:.2f}</text>')
    lines.append('</svg>')
    path.write_text('\n'.join(lines), encoding='utf-8')
    return path


def write_svg_line(path: Path, title: str, y_values: list[float], color: str = '#059669') -> Path:
    width, height = 980, 360
    left, top, right, bottom = 70, 50, 40, 50
    plot_w = width - left - right
    plot_h = height - top - bottom

    if not y_values:
        y_values = [0.0]
    y_min, y_max = min(y_values), max(y_values)
    y_span = max(y_max - y_min, 1e-6)

    def xy(i: int, y: float) -> tuple[float, float]:
        x = left + (i / max(len(y_values) - 1, 1)) * plot_w
        py = top + (1.0 - ((y - y_min) / y_span)) * plot_h
        return x, py

    points = [xy(i, y) for i, y in enumerate(y_values)]
    path_d = ' '.join([('M' if i == 0 else 'L') + f' {x:.2f} {y:.2f}' for i, (x, y) in enumerate(points)])

    lines = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<style>text{font-family:Segoe UI,Arial,sans-serif;font-size:12px;fill:#334155}.title{font-size:20px;font-weight:700;fill:#0f172a}</style>',
        f'<rect width="{width}" height="{height}" fill="#f8fafc"/>',
        f'<text x="20" y="30" class="title">{title}</text>',
        f'<rect x="{left}" y="{top}" width="{plot_w}" height="{plot_h}" fill="#ffffff" stroke="#cbd5e1"/>',
        f'<path d="{path_d}" stroke="{color}" stroke-width="3" fill="none"/>',
    ]
    for i, (x, y) in enumerate(points):
        lines.append(f'<circle cx="{x:.2f}" cy="{y:.2f}" r="3" fill="{color}"/>')
        if i % max(len(points) // 8, 1) == 0:
            lines.append(f'<text x="{x:.2f}" y="{height - 20}" text-anchor="middle">{i}</text>')
    lines.append('</svg>')
    path.write_text('\n'.join(lines), encoding='utf-8')
    return path


def dijkstra_costs(edges: list[dict], origin: str) -> dict[str, float]:
    adj: dict[str, list[tuple[str, float]]] = {}
    for e in edges:
        a, b = str(e['from_node']), str(e['to_node'])
        c = float(e['cost'])
        adj.setdefault(a, []).append((b, c))
        adj.setdefault(b, []).append((a, c))

    dist = {origin: 0.0}
    pq: list[tuple[float, str]] = [(0.0, origin)]
    while pq:
        cd, node = heappop(pq)
        if cd > dist.get(node, float('inf')):
            continue
        for nxt, w in adj.get(node, []):
            nd = cd + w
            if nd < dist.get(nxt, float('inf')):
                dist[nxt] = nd
                heappush(pq, (nd, nxt))
    return dist


from geoprompt.network.routing import build_network_graph, service_area
from geoprompt.network import closest_facility
from geoprompt.network.utility import outage_impact_report, restoration_sequence_report

## Section A: Pull Utilities Data Sources


In [8]:
# Real connector stubs + large deterministic fallback payloads.
overpass_fallback = {
    'elements': [
        {'type': 'node', 'id': i, 'lat': 40.70 + (i % 10) * 0.008, 'lon': -111.95 + (i // 10) * 0.008}
        for i in range(1, 81)
    ]
}
census_fallback = {
    'NAME': 'Salt Lake County, Utah',
    'B01001_001E': '1181213',
    'B19013_001E': '83717',
    'B25064_001E': '1512',
}
noaa_fallback = {
    'properties': {
        'periods': [
            {'name': f'Period {i}', 'temperature': 48 + (i % 7), 'shortForecast': 'Partly Cloudy'}
            for i in range(1, 15)
        ]
    }
}

overpass = fetch_json('https://overpass-api.de/api/interpreter?data=[out:json];way[%22highway%22](40.70,-111.95,40.80,-111.80);(._;>;);out body;', overpass_fallback)
census = fetch_json('https://api.census.gov/data/2022/acs/acs5?get=NAME,B01001_001E,B19013_001E,B25064_001E&for=county:035&in=state:49', census_fallback)
noaa = fetch_json('https://api.weather.gov/points/40.7608,-111.8910', noaa_fallback)

source_summary = pd.DataFrame(
    [
        {'source': 'Overpass', 'records': len(overpass.get('elements', [])), 'live_api': ALLOW_LIVE_API},
        {'source': 'Census ACS', 'records': len(census.keys()) if isinstance(census, dict) else 0, 'live_api': ALLOW_LIVE_API},
        {'source': 'NOAA', 'records': len(noaa.get('properties', {}).get('periods', [])), 'live_api': ALLOW_LIVE_API},
    ]
)
print('Data source ingest summary')
display(source_summary)

Data source ingest summary


,source,records,live_api
0,Overpass,80,False
1,Census ACS,4,False
2,NOAA,14,False


## Section B: Simple Track


In [9]:
# Build a denser synthetic utility network (36 nodes, 60+ edges) for richer analysis.
coords: dict[str, tuple[float, float]] = {}
for r in range(6):
    for c in range(6):
        node = f'N{r}_{c}'
        coords[node] = (c * 1.0, r * 1.0)
coords['SRC'] = (-1.0, 2.5)

edges = []
edge_id = 1
for r in range(6):
    for c in range(6):
        node = f'N{r}_{c}'
        if c < 5:
            right = f'N{r}_{c+1}'
            edges.append({'edge_id': f'e{edge_id}', 'from_node': node, 'to_node': right, 'cost': 1.0 + (r % 2) * 0.15})
            edge_id += 1
        if r < 5:
            down = f'N{r+1}_{c}'
            edges.append({'edge_id': f'e{edge_id}', 'from_node': node, 'to_node': down, 'cost': 1.0 + (c % 3) * 0.1})
            edge_id += 1

for r in range(6):
    node = f'N{r}_0'
    edges.append({'edge_id': f'e{edge_id}', 'from_node': 'SRC', 'to_node': node, 'cost': 1.4 + r * 0.08})
    edge_id += 1

graph = build_network_graph(edges, directed=False)

incidents = [(random.uniform(0, 5), random.uniform(0, 5)) for _ in range(80)]
facilities = [(0.0, 0.0), (5.0, 0.0), (0.0, 5.0), (5.0, 5.0), (2.5, 2.5), (-1.0, 2.5)]
nearest = closest_facility(incidents, facilities, n_closest=1)

area = service_area(graph, origins=['SRC'], max_cost=4.5)
costs = dijkstra_costs(edges, 'SRC')

simple_df = pd.DataFrame(nearest)
service_df = pd.DataFrame(area)
service_df['node_cost_ref'] = service_df['node'].map(costs)
service_df['cost_delta_abs'] = (service_df['cost'] - service_df['node_cost_ref']).abs().round(8)

print('Nearest-facility sample (first 12 rows)')
display(simple_df.head(12))
print('Service-area with reference cost comparison (first 20 rows)')
display(service_df.sort_values('cost').head(20))

simple_payload = {
    'track': 'simple',
    'nearest_facility_pairs': nearest,
    'service_area': area,
    'network_edge_count': len(edges),
    'node_count': len(coords),
    'max_cost_delta': float(service_df['cost_delta_abs'].max()) if not service_df.empty else 0.0,
}
(simple_path := OUTPUT_DIR / 'd1-notebook-simple.json').write_text(json.dumps(simple_payload, indent=2), encoding='utf-8')
print('Wrote', simple_path)

Nearest-facility sample (first 12 rows)


,incident_index,facility_index,distance,rank
0,0,4,2.150503,1
1,1,4,1.312065,1
2,2,0,1.625404,1
3,3,3,1.363199,1
4,4,4,0.937076,1
5,5,0,1.249193,1
6,6,4,1.618636,1
7,7,3,0.644573,1
8,8,4,1.345512,1
9,9,4,0.945000,1


Service-area with reference cost comparison (first 20 rows)


,node,assigned_origin,cost,within_service_area,node_cost_ref,cost_delta_abs
0,SRC,SRC,0.00,True,0.00,0.0
1,N0_0,SRC,1.40,True,1.40,0.0
2,N1_0,SRC,1.48,True,1.48,0.0
3,N2_0,SRC,1.56,True,1.56,0.0
4,N3_0,SRC,1.64,True,1.64,0.0
5,N4_0,SRC,1.72,True,1.72,0.0
6,N5_0,SRC,1.80,True,1.80,0.0
7,N0_1,SRC,2.40,True,2.40,0.0
8,N2_1,SRC,2.56,True,2.56,0.0
9,N1_1,SRC,2.63,True,2.63,0.0


Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d1-notebook-simple.json


## Section B2: Map And Plot Outputs (Simple Track)

In [10]:
# Produce multiple user-facing outputs: map-like GeoJSON + SVG plot bundle.
service_nodes = service_df[['node', 'cost', 'node_cost_ref', 'cost_delta_abs']].copy()
service_nodes['x'] = service_nodes['node'].map(lambda n: coords.get(str(n), (0.0, 0.0))[0])
service_nodes['y'] = service_nodes['node'].map(lambda n: coords.get(str(n), (0.0, 0.0))[1])

geojson = {
    'type': 'FeatureCollection',
    'features': [
        {
            'type': 'Feature',
            'properties': {
                'node': row['node'],
                'cost': float(row['cost']),
                'ref_cost': float(row['node_cost_ref']),
                'delta': float(row['cost_delta_abs']),
            },
            'geometry': {'type': 'Point', 'coordinates': [float(row['x']), float(row['y'])]},
        }
        for _, row in service_nodes.iterrows()
    ],
}
geojson_path = OUTPUT_DIR / 'd1-notebook-service-area.geojson'
geojson_path.write_text(json.dumps(geojson, indent=2), encoding='utf-8')

# Simple self-contained HTML map-like view.
map_html = [
    '<!doctype html><html><head><meta charset="utf-8"><title>D1 Service Area Map</title>',
    '<style>body{font-family:Segoe UI,Arial,sans-serif;margin:24px;} .pt{position:absolute;border-radius:50%;width:10px;height:10px;background:#2563eb;} .canvas{position:relative;width:720px;height:420px;border:1px solid #cbd5e1;background:#f8fafc;}</style>',
    '</head><body><h1>D1 Service Area Node Map</h1><div class="canvas">',
]
for _, row in service_nodes.iterrows():
    left = 40 + float(row['x']) * 110
    top = 30 + float(row['y']) * 60
    title = f"{row['node']} cost={float(row['cost']):.2f}"
    map_html.append(f'<div class="pt" title="{title}" style="left:{left:.1f}px;top:{top:.1f}px"></div>')
map_html.append('</div></body></html>')
map_html_path = OUTPUT_DIR / 'd1-notebook-service-area-map.html'
map_html_path.write_text('\n'.join(map_html), encoding='utf-8')

# Plot 1: top service cost bars.
top_cost_rows = list(service_nodes.sort_values('cost', ascending=False).head(20)[['node', 'cost']].itertuples(index=False, name=None))
bar_path = write_svg_bar(OUTPUT_DIR / 'd1-notebook-top-service-costs.svg', 'Top 20 Service Costs', top_cost_rows, color='#dc2626')

# Plot 2: nearest distance by incident index.
nearest_line = [float(v) for v in simple_df['distance'].tolist()]
line_path = write_svg_line(OUTPUT_DIR / 'd1-notebook-nearest-distance-trend.svg', 'Nearest Facility Distance Trend', nearest_line, color='#0ea5e9')

print('Wrote', geojson_path)
print('Wrote', map_html_path)
print('Wrote', bar_path)
print('Wrote', line_path)

Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d1-notebook-service-area.geojson
Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d1-notebook-service-area-map.html
Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d1-notebook-top-service-costs.svg
Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d1-notebook-nearest-distance-trend.svg


## Section C: Complex Track


In [11]:
# Richer complex simulation with multiple disruption modes and larger scenario set.
rng = random.Random(42)
edge_ids = [str(e['edge_id']) for e in edges]
node_names = [n for n in coords if n != 'SRC']

demand_by_node = {n: round(8.0 + ((i * 3) % 17) + (i % 5) * 0.4, 2) for i, n in enumerate(node_names)}
demand_by_node['SRC'] = 0.0
critical_nodes = [n for i, n in enumerate(node_names) if i % 6 == 0][:8]

outage_rows = []
for run in range(120):
    failed_count = 1 if run % 3 else 2
    failed_edges = rng.sample(edge_ids, k=failed_count)
    outage = outage_impact_report(
        graph,
        source_nodes=['SRC'],
        failed_edges=failed_edges,
        demand_by_node=demand_by_node,
        critical_nodes=critical_nodes,
        outage_hours=2.0 + (run % 4) * 0.5,
    )
    outage_rows.append({'run': run, 'failed_edges': failed_edges, **outage})

# Restoration baseline for top two common failures.
popular_failed = [edge_ids[0], edge_ids[1]]
restoration = restoration_sequence_report(
    graph,
    source_nodes=['SRC'],
    failed_edges=popular_failed,
    demand_by_node=demand_by_node,
    critical_nodes=critical_nodes,
)

outage_df = pd.DataFrame(outage_rows)
outage_df['cost_bucket'] = pd.cut(
    outage_df['estimated_cost'],
    bins=[-1, 80, 120, 170, 999999],
    labels=['low', 'moderate', 'high', 'critical'],
)
outage_df['reliability_index'] = (1.0 - (outage_df['severity_score'] / (outage_df['severity_score'].max() + 1e-9))).round(4)

scenario_table = outage_df.groupby('cost_bucket', observed=False).agg(
    runs=('run', 'count'),
    avg_cost=('estimated_cost', 'mean'),
    avg_impacted_demand=('impacted_demand', 'mean'),
    avg_reliability=('reliability_index', 'mean'),
).reset_index()

display(outage_df[['run', 'estimated_cost', 'severity_tier', 'impacted_demand', 'reliability_index']].head(25))
print('Scenario cost bucket summary')
display(scenario_table)

complex_payload = {
    'track': 'complex',
    'monte_carlo_runs': int(len(outage_rows)),
    'avg_estimated_outage_cost': float(round(outage_df['estimated_cost'].mean(), 2)),
    'p95_estimated_outage_cost': float(round(outage_df['estimated_cost'].quantile(0.95), 2)),
    'avg_reliability_index': float(round(outage_df['reliability_index'].mean(), 4)),
    'outage_rows': outage_rows,
    'restoration': restoration,
    'critical_nodes': critical_nodes,
    'demand_by_node': demand_by_node,
}
(complex_path := OUTPUT_DIR / 'd1-notebook-complex.json').write_text(json.dumps(complex_payload, indent=2), encoding='utf-8')

print('Complex outage scenario results (expanded)')
print('Wrote', complex_path)

,run,estimated_cost,severity_tier,impacted_demand,reliability_index
0,0,0.0,low,0,1.0
1,1,0.0,low,0,1.0
2,2,0.0,low,0,1.0
3,3,0.0,low,0,1.0
4,4,0.0,low,0,1.0
5,5,0.0,low,0,1.0
6,6,0.0,low,0,1.0
7,7,0.0,low,0,1.0
8,8,0.0,low,0,1.0
9,9,0.0,low,0,1.0


Scenario cost bucket summary


,cost_bucket,runs,avg_cost,avg_impacted_demand,avg_reliability
0,low,120,0.0,0.0,1.0
1,moderate,0,NaN,NaN,NaN
2,high,0,NaN,NaN,NaN
3,critical,0,NaN,NaN,NaN


Complex outage scenario results (expanded)
Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d1-notebook-complex.json


## Section C2: Advanced Plots, Heatmap, And Executive Outputs

In [12]:
# Plot bundle for complex analysis.

# 1) Outage cost trend line.
cost_line_path = write_svg_line(
    OUTPUT_DIR / 'd1-notebook-outage-cost-trend.svg',
    'Monte Carlo Estimated Outage Cost Trend (120 runs)',
    [float(v) for v in outage_df['estimated_cost'].tolist()],
    color='#b91c1c',
)

# 2) Reliability by severity bucket.
bucket_rows = [
    (str(row['cost_bucket']), float(row['avg_reliability']))
    for _, row in scenario_table.sort_values('avg_reliability', ascending=False).iterrows()
    if pd.notna(row['avg_reliability'])
]
if not bucket_rows:
    bucket_rows = [('no-data', 0.0)]

reliability_bar_path = write_svg_bar(
    OUTPUT_DIR / 'd1-notebook-reliability-by-bucket.svg',
    'Average Reliability By Cost Bucket',
    bucket_rows,
    color='#16a34a',
)

# 3) Heatmap-like SVG (node demand + criticality).
heat_w, heat_h = 900, 440
heat_lines = [
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{heat_w}" height="{heat_h}" viewBox="0 0 {heat_w} {heat_h}">',
    '<style>text{font-family:Segoe UI,Arial,sans-serif;font-size:11px;fill:#0f172a}.title{font-size:18px;font-weight:700}</style>',
    f'<rect width="{heat_w}" height="{heat_h}" fill="#f8fafc"/>',
    '<text x="18" y="28" class="title">Critical Demand Heatmap (Synthetic Utility Grid)</text>',
]
for node, (x, y) in coords.items():
    if node == 'SRC':
        continue
    demand = float(demand_by_node.get(node, 0.0))
    intensity = min(max((demand - 8.0) / 20.0, 0.0), 1.0)
    red = int(255 * intensity)
    blue = int(255 * (1 - intensity))
    fill = f'rgb({red},80,{blue})'
    left = 80 + x * 120
    top = 60 + y * 55
    border = '#111827' if node in critical_nodes else '#cbd5e1'
    stroke_w = 2 if node in critical_nodes else 1
    heat_lines.append(f'<rect x="{left:.1f}" y="{top:.1f}" width="42" height="28" rx="4" fill="{fill}" stroke="{border}" stroke-width="{stroke_w}"/>')
    heat_lines.append(f'<text x="{left + 21:.1f}" y="{top + 18:.1f}" text-anchor="middle" fill="#ffffff">{node.split("_")[1]}</text>')
heat_lines.append('</svg>')
heatmap_path = OUTPUT_DIR / 'd1-notebook-demand-heatmap.svg'
heatmap_path.write_text('\n'.join(heat_lines), encoding='utf-8')

# 4) Executive markdown summary without optional dependencies.
bucket_csv_lines = ['cost_bucket,runs,avg_cost,avg_impacted_demand,avg_reliability']
for _, row in scenario_table.iterrows():
    bucket_csv_lines.append(
        f"{row['cost_bucket']},{int(row['runs']) if pd.notna(row['runs']) else 0},{float(row['avg_cost']) if pd.notna(row['avg_cost']) else 0.0:.2f},{float(row['avg_impacted_demand']) if pd.notna(row['avg_impacted_demand']) else 0.0:.2f},{float(row['avg_reliability']) if pd.notna(row['avg_reliability']) else 0.0:.4f}"
    )

summary_md = '\n'.join([
    '# D1 Utilities Expanded Executive Summary',
    '',
    f"- Monte Carlo runs: {len(outage_df)}",
    f"- Average outage cost: {outage_df['estimated_cost'].mean():.2f}",
    f"- P95 outage cost: {outage_df['estimated_cost'].quantile(0.95):.2f}",
    f"- Average reliability index: {outage_df['reliability_index'].mean():.4f}",
    f"- Critical nodes tracked: {len(critical_nodes)}",
    '',
    '## Cost Bucket Table (CSV)',
    '',
    *bucket_csv_lines,
])
summary_path = OUTPUT_DIR / 'd1-notebook-executive-summary.md'
summary_path.write_text(summary_md, encoding='utf-8')

print('Wrote', cost_line_path)
print('Wrote', reliability_bar_path)
print('Wrote', heatmap_path)
print('Wrote', summary_path)
print('Top rows of outage table:')
display(outage_df[['run', 'estimated_cost', 'cost_bucket', 'reliability_index']].head(12))

Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d1-notebook-outage-cost-trend.svg
Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d1-notebook-reliability-by-bucket.svg
Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d1-notebook-demand-heatmap.svg
Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d1-notebook-executive-summary.md
Top rows of outage table:


,run,estimated_cost,cost_bucket,reliability_index
0,0,0.0,low,1.0
1,1,0.0,low,1.0
2,2,0.0,low,1.0
3,3,0.0,low,1.0
4,4,0.0,low,1.0
5,5,0.0,low,1.0
6,6,0.0,low,1.0
7,7,0.0,low,1.0
8,8,0.0,low,1.0
9,9,0.0,low,1.0
